In [ ]:
import pandas as pd
import numpy as np


from openai import OpenAI
df= pd.read_csv('top_rated_wines.csv' )
df = df[df["variety"].notna()]
data= df.sample(frac=0.5).to_dict(orient='records')
print(data)


[{'name': 'Joseph Phelps Insignia (1.5 Liter Magnum) 2014', 'region': 'Napa Valley, California', 'variety': 'Red Wine', 'rating': 96.0, 'notes': 'The 2014 Insignia opens with expressive blackberry, cocoa powder, Bergamot and floral aromatics. Fresh and focused with concentrated black fruit, mocha, cardamom and Madagascar vanilla notes. Opulent and rich with supple tannin structure and balance.'}, {'name': 'Aubert Larry Hyde & Sons Vineyard Chardonnay (1.5 Liter) 2014', 'region': 'Carneros, California', 'variety': 'White Wine', 'rating': 97.0, 'notes': 'There is a greenish hue on the edge of the glass, which is indicative of the health of the wine. Initially, aromatics of white nectarines and green apple showcase delicate fruits. Secondary aromas evolve with time to form more expressive, complex notes of bergamot, sea air and citrus flowers. The palate is racy, yet rich, with intense gras and oily textures. Layered with extract and acid, the 2014 Larry Hyde & Sons has what it takes to a

In [ ]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

ModuleNotFoundError: No module named 'qdrant_client'

In [ ]:

encoder = SentenceTransformer("all-MiniLM-L6-v2") # Model to create embeddings

In [ ]:
quadrant= QdrantClient(":memory:") # Create in-memory Qdrant instance

In [ ]:
# Create collection to store wines
quadrant.recreate_collection(
    collection_name="top_wines",
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

In [ ]:
quadrant.upload_points(
    collection_name="top_wines", 
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(docs["notes"]).tolist(),# Create embedding for wine variety
            payload=docs,
        )
        for idx, docs in enumerate(data) # data is the variable holding all the wines
    ]
)

In [ ]:
user_prompt = "Which wines are similar to a fruity and soft Pinot Noir?"

In [ ]:
hits = quadrant.search(
        collection_name="top_wines",
        query_vector=encoder.encode(user_prompt).tolist(), # Create embedding for user prompt
        limit=3, # Return 3 most similar wines
)
for hit in hits:
    payload = (hit.payload,"score: ", hit.score)
    print(payload)

In [ ]:
search_result = [hit.payload for hit in hits]

In [ ]:
from openai import OpenAI
client = OpenAI(
    base_url="http://127.0.0.1:8080/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=[
        {"role": "system", "content": "You are chatbot, a wine specialist. Your top priority is to help guide users into selecting amazing wine and guide them with their requests."},
        {"role": "user", "content": "Suggest me an amazing Malbec wine from Argentina"},
        {"role": "assistant", "content": str(search_result)}
    ]
)
print(completion.choices[0].message)